In [0]:
%run ./utils_dq

In [0]:
dbutils.widgets.text("run_date", "", "Run Date (YYYY-MM-DD)")

In [0]:
run_date = dbutils.widgets.get("run_date")

BRONZE = "abfss://lakehouse@saatliqcommerce01dl.dfs.core.windows.net/bronze"

src_path = f"{BRONZE}/payments/ingest_date={run_date}"

payments_raw = spark.read.parquet(src_path)

print("Bronze batch rows:", payments_raw.count())
print(
    "Unique payment IDs:",
    payments_raw.select("payment_id").distinct().count()
)

Bronze batch rows: 21
Unique payment IDs: 21


In [0]:

from pyspark.sql import functions as F

BRONZE = "abfss://lakehouse@saatliqcommerce01dl.dfs.core.windows.net/bronze"

src_path = f"{BRONZE}/payments/ingest_date={run_date}"

df_payments = spark.read.parquet(src_path)

# Understand columns and data types
df_payments.printSchema()

# Check total records and payment IDs
display(
    df_payments.agg(
        F.count("*").alias("total_rows"),
        F.countDistinct("payment_id").alias("unique_payment_ids"),
        F.sum(F.col("payment_id").isNull().cast("int")).alias("null_payment_ids")
    )
)

# Identify payment-method variations
display(
    df_payments.groupBy("method")
    .count()
    .orderBy("method")
)

# Inspect actual payment records
display(df_payments.limit(10))


root
 |-- payment_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- amount: decimal(12,2) (nullable = true)
 |-- method: string (nullable = true)
 |-- paid_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)



total_rows,unique_payment_ids,null_payment_ids
21,21,0


method,count
COD,4
Credit Card,2
DebitCard,1
Net Banking,5
UPI,4
Wallet,3
credit card,1
net banking,1


payment_id,order_id,amount,method,paid_at,updated_at
8363,10041,1998.00,Net Banking,2026-10-31T17:40:19.000Z,2026-10-31T17:40:19.000Z
8364,10042,27791.00,UPI,2026-10-31T17:40:19.000Z,2026-10-31T17:40:19.000Z
8365,10043,23694.00,Net Banking,2026-10-31T17:40:19.000Z,2026-10-31T17:40:19.000Z
8366,10044,7147.00,net banking,2026-10-31T17:40:19.000Z,2026-10-31T17:40:19.000Z
8367,10045,4794.00,Wallet,2026-10-31T17:40:19.000Z,2026-10-31T17:40:19.000Z
8368,10046,26340.00,COD,2026-10-31T17:40:19.000Z,2026-10-31T17:40:19.000Z
8369,10047,9696.00,Credit Card,2026-10-31T17:40:19.000Z,2026-10-31T17:40:19.000Z
8370,10048,8392.00,Net Banking,2026-10-31T17:40:19.000Z,2026-10-31T17:40:19.000Z
8371,10049,11297.00,Wallet,2026-10-31T17:40:19.000Z,2026-10-31T17:40:19.000Z
8372,10050,5397.00,Net Banking,2026-10-31T17:40:19.000Z,2026-10-31T17:40:19.000Z


In [0]:

from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

# Map all observed variations to the six approved methods
METHOD_MAP = {
    "upi": "UPI",
    "credit card": "Credit Card",
    "creditcard": "Credit Card",
    "debit card": "Debit Card",
    "debitcard": "Debit Card",
    "net banking": "Net Banking",
    "netbanking": "Net Banking",
    "wallet": "Wallet",
    "cod": "COD",
    "cash on delivery": "COD"
}

method_map = F.create_map(
    *[F.lit(x) for pair in METHOD_MAP.items() for x in pair]
)

# Normalize the lookup key
method_key = F.lower(
    F.regexp_replace(
        F.trim(F.col("method")), r"\s+", " "
    )
)

# Clean the payments
payments_clean = (
    df_payments
    .withColumn("method", method_map[method_key])
    .withColumn("amount", F.col("amount").cast("decimal(12,2)"))
    .filter(F.col("payment_id").isNotNull())
)

# Keep the latest version of each payment_id
w = Window.partitionBy("payment_id").orderBy(
    F.col("updated_at").desc_nulls_last()
)

payments_clean = (
    payments_clean
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

display(
    payments_clean.groupBy("method")
    .count()
    .orderBy("method")
)


method,count
COD,4
Credit Card,3
Debit Card,1
Net Banking,6
UPI,4
Wallet,3


In [0]:

# Dynamic validation - no hardcoded row count

clean_count = payments_clean.count()

unique_ids = payments_clean.select(
    "payment_id"
).distinct().count()

null_ids = df_payments.filter(
    F.col("payment_id").isNull()
).count()

invalid_rows = payments_clean.filter(
    F.col("order_id").isNull() |
    F.col("method").isNull() |
    F.col("amount").isNull() |
    (F.col("amount") <= 0) |
    F.col("paid_at").isNull() |
    F.col("updated_at").isNull()
).count()

print("Cleaned payments:", clean_count)
print("Unique payment IDs:", unique_ids)
print("NULL payment IDs:", null_ids)
print("Invalid rows:", invalid_rows)

assert clean_count == unique_ids, "Duplicate payment IDs found"
assert null_ids == 0, "NULL payment IDs found"
assert invalid_rows == 0, "Invalid payment records found"

print("Payments validation PASSED!")


Cleaned payments: 21
Unique payment IDs: 21
NULL payment IDs: 0
Invalid rows: 0
Payments validation PASSED!


In [0]:

TARGET = "atliq_dl.silver.payments"

# Create target Delta table if it doesn't exist
(
    DeltaTable.createIfNotExists(spark)
    .tableName(TARGET)
    .addColumns(payments_clean.schema)
    .execute()
)

# Insert new records and update newer versions
(
    DeltaTable.forName(spark, TARGET)
    .alias("t")
    .merge(
        payments_clean.alias("s"),
        "t.payment_id = s.payment_id"
    )
    .whenMatchedUpdateAll(
        condition="s.updated_at > t.updated_at"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("Payments MERGE completed!")

display(
    spark.table(TARGET).agg(
        F.count("*").alias("silver_rows"),
        F.countDistinct("payment_id").alias("unique_payment_ids")
    )
)


Payments MERGE completed!


silver_rows,unique_payment_ids
8383,8383


In [0]:
from pyspark.sql import Window

raw = spark.read.parquet(
    f"{BRONZE}/payments/ingest_date={dbutils.widgets.get('run_date')}"
)

print("DQ batch rows:", raw.count())
silver = spark.table("atliq_dl.silver.payments")

w = Window.partitionBy("payment_id").orderBy(
    F.col("updated_at").desc()
)

latest = (
    raw.withColumn("_dq_rn", F.row_number().over(w))
       .filter(F.col("_dq_rn") == 1)
       .drop("_dq_rn")
)

j = (
    latest.alias("r")
    .join(silver.alias("s"), "payment_id")
    .filter(F.col("r.updated_at").eqNullSafe(F.col("s.updated_at")))
)

log_dq("payments", "payment_methods_standardized",
       j.filter(
           ~F.col("r.method").eqNullSafe(F.col("s.method"))
       ).count())

log_dq("payments", "multiple_payment_versions_collapsed",
       raw.count() - latest.count())

log_dq("payments", "repeated_order_payment_ids_retained",
       latest.count() - latest.dropDuplicates(["order_id"]).count())

display(
    spark.table("atliq_dl.silver.dq_log")
    .filter("table_name = 'payments'")
    .orderBy("rule")
)

DQ batch rows: 21


run_date,table_name,rule,rows_affected
2026-11-01,payments,multiple_payment_versions_collapsed,0
2026-10-31,payments,multiple_payment_versions_collapsed,0
2026-11-01,payments,payment_methods_standardized,3
2026-10-31,payments,payment_methods_standardized,1015
2026-10-31,payments,repeated_order_payment_ids_retained,110
2026-11-01,payments,repeated_order_payment_ids_retained,1
